# Radar Systems — Interactive Notebook

Radar (**Ra**dio **D**etection **A**nd **R**anging) measures the position and velocity of objects
by transmitting electromagnetic pulses and analysing their echoes.
This notebook builds up the core concepts from first principles,
each one backed by an interactive visualisation you can manipulate in real time.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.signal import correlate

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

c = 3e8  # speed of light [m/s]
print("Environment ready  —  c = 3 × 10⁸ m/s")

Environment ready  —  c = 3 × 10⁸ m/s


## 1 — Radar Echo and Range Measurement

A radar transmits a short pulse of duration $\tau$.
The pulse travels at speed $c$, hits a target at range $R$, and returns after a round-trip time

$$\Delta t = \frac{2R}{c}  \quad\Longrightarrow\quad  R = \frac{c\,\Delta t}{2}$$

The visualisation below has **two panels**:
* **Top** — a physical side-view: the radar is on the left, the target on the right. A pulse packet travels out, bounces, and returns. Use the **time slider** to step through the event.
* **Bottom** — the signal timeline showing when the transmit burst and the echo appear.

In [3]:
out1 = widgets.Output()

range_sl = widgets.FloatSlider(value=30, min=5, max=100, step=0.5,
    description="Target R [km]:", style={"description_width": "110px"},
    layout=widgets.Layout(width="560px"))
time_sl = widgets.FloatSlider(value=0, min=0, max=1.0, step=0.005,
    description="Time [ms]:", style={"description_width": "110px"},
    layout=widgets.Layout(width="560px"),
    readout_format=".3f")

def update_echo(change=None):
    R = range_sl.value * 1e3
    tau = 5e-6
    dt_rt = 2 * R / c
    t_max = max(dt_rt * 1.4, 0.3e-3)
    time_sl.max = round(t_max * 1e3, 3)
    t_now = time_sl.value * 1e-3
    pulse_len_m = c * tau
    if t_now <= tau:
        pulse_front = c * t_now
        pulse_back = 0
        phase = "transmitting"
    elif t_now <= dt_rt / 2:
        pulse_front = c * t_now
        pulse_back = pulse_front - pulse_len_m
        phase = "outbound"
    elif t_now <= dt_rt / 2 + tau:
        pulse_front = min(c * t_now, R)
        pulse_back = pulse_front - pulse_len_m
        phase = "hitting target"
    elif t_now <= dt_rt:
        travelled_back = c * (t_now - dt_rt / 2) - R
        pulse_front_pos = R - (c * (t_now - dt_rt / 2) - R)
        pulse_back_pos = pulse_front_pos + pulse_len_m
        pulse_front = min(pulse_front_pos, R)
        pulse_back = max(pulse_front - pulse_len_m, 0)
        echo_front = R - c * (t_now - dt_rt / 2)
        echo_back = echo_front + pulse_len_m
        phase = "echo returning"
    elif t_now <= dt_rt + tau:
        echo_front = R - c * (t_now - dt_rt / 2)
        echo_back = echo_front + pulse_len_m
        phase = "echo arriving"
    else:
        phase = "echo received"

    with out1:
        clear_output(wait=True)
        fig, axes = plt.subplots(2, 1, figsize=(11, 5.5),
                                 gridspec_kw={"height_ratios": [1.2, 1]})
        ax = axes[0]
        ax.set_xlim(-R * 0.08, R * 1.15)
        ax.set_ylim(-1, 1)
        ax.set_yticks([])
        ax.set_xlabel("Distance [km]")
        ax.set_title(f"Physical view  —  {phase}", fontsize=11)
        ax.plot(0, 0, "s", color="C0", ms=18, zorder=5)
        ax.annotate("RADAR", (0, -0.55), ha="center", fontsize=9, color="C0", weight="bold")
        ax.plot(R, 0, "D", color="C3", ms=16, zorder=5)
        ax.annotate(f"TARGET\n{R/1e3:.1f} km", (R, -0.55), ha="center", fontsize=9, color="C3")
        ax.axhline(0, color="gray", lw=0.4, alpha=0.5)

        if t_now <= dt_rt / 2 + tau:
            pf = min(c * t_now, R)
            pb = max(pf - pulse_len_m, 0)
            ax.axvspan(pb, pf, ymin=0.4, ymax=0.6, color="C0", alpha=0.7)
            ax.annotate("Tx pulse", (pf, 0.35), fontsize=8, color="C0")
        if t_now > dt_rt / 2:
            ef = R - c * (t_now - dt_rt / 2) + R
            ef = 2 * R - c * (t_now - dt_rt / 2)
            eb = ef + pulse_len_m
            ef_clip = max(ef, 0)
            eb_clip = min(eb, R)
            if eb_clip > ef_clip:
                ax.axvspan(ef_clip, eb_clip, ymin=0.4, ymax=0.6, color="C3", alpha=0.7)
                ax.annotate("Echo", (ef_clip, -0.35), fontsize=8, color="C3")
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e3:.0f}"))

        ax2 = axes[1]
        t_sig = np.linspace(0, t_max, 4000)
        tx_env = np.where((t_sig >= 0) & (t_sig <= tau), 1.0, 0.0)
        rx_env = np.where((t_sig >= dt_rt) & (t_sig <= dt_rt + tau), 0.6, 0.0)
        ax2.fill_between(t_sig * 1e6, tx_env, alpha=0.4, color="C0", label="Tx pulse")
        ax2.fill_between(t_sig * 1e6, rx_env, alpha=0.5, color="C3", label="Echo")
        ax2.axvline(t_now * 1e6, color="k", ls="--", lw=1.2, label="current time")
        ax2.annotate("", xy=(dt_rt * 1e6, 0.75), xytext=(0, 0.75),
                     arrowprops=dict(arrowstyle="<->", color="k", lw=1.3))
        ax2.text(dt_rt * 1e6 / 2, 0.82, f"Δt = {dt_rt*1e6:.1f} µs → R = {R/1e3:.1f} km",
                 ha="center", fontsize=9, bbox=dict(fc="white", ec="gray", pad=2))
        ax2.set_xlabel("Time [µs]")
        ax2.set_ylabel("Amplitude")
        ax2.set_title("Signal timeline")
        ax2.set_ylim(-0.1, 1.15)
        ax2.legend(loc="upper right", fontsize=8)
        plt.tight_layout()
        plt.show()

range_sl.observe(update_echo, "value")
time_sl.observe(update_echo, "value")
display(widgets.VBox([range_sl, time_sl]), out1)
update_echo()

Output()

## 2 — Pulse Repetition Interval, PRF and Duty Cycle

| Parameter | Symbol | Relation |
|---|---|---|
| Pulse Repetition Frequency | $\text{PRF} = 1/\text{PRI}$ | Sets how often the radar "listens" |
| Max unambiguous range | $R_{\max} = c \cdot \text{PRI}\;/\;2$ | Echoes arriving later fold back |
| Duty cycle | $\delta = \tau / \text{PRI}$ | Fraction of time the transmitter is on |

A **higher PRF** updates faster but **shrinks** $R_{\max}$.

In [4]:
out2 = widgets.Output()

prf_slider = widgets.FloatSlider(value=1000, min=200, max=5000, step=50,
    description="PRF [Hz]:", style={"description_width": "100px"},
    layout=widgets.Layout(width="560px"))
pw_slider = widgets.FloatSlider(value=5, min=0.5, max=30, step=0.5,
    description="Pulse τ [µs]:", style={"description_width": "100px"},
    layout=widgets.Layout(width="560px"))

def update_pri(change=None):
    prf = prf_slider.value
    tau = pw_slider.value * 1e-6
    pri = 1.0 / prf
    r_max = c * pri / 2
    duty = tau / pri * 100
    n_pulses = 5
    t = np.linspace(0, n_pulses * pri, 8000)
    pulse_train = np.zeros_like(t)
    for i in range(n_pulses):
        mask = (t >= i * pri) & (t < i * pri + tau)
        pulse_train[mask] = 1.0
    with out2:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 3))
        ax.fill_between(t * 1e3, pulse_train, step="mid", alpha=0.5, color="C0")
        ax.step(t * 1e3, pulse_train, where="mid", color="C0", lw=1.2)
        for i in range(n_pulses):
            ax.annotate("", xy=((i + 1) * pri * 1e3, -0.15),
                         xytext=(i * pri * 1e3, -0.15),
                         arrowprops=dict(arrowstyle="<->", color="C1", lw=1.2))
        ax.text(pri * 1e3 / 2, -0.28, f"PRI = {pri*1e6:.0f} µs", ha="center", fontsize=9, color="C1")
        info = (f"PRF = {prf:.0f} Hz  |  PRI = {pri*1e6:.0f} µs  |  "
                f"R_max = {r_max/1e3:.1f} km  |  Duty = {duty:.2f}%")
        ax.set_title(info, fontsize=10)
        ax.set_xlabel("Time [ms]")
        ax.set_ylabel("Amplitude")
        ax.set_ylim(-0.4, 1.3)
        plt.tight_layout()
        plt.show()

prf_slider.observe(update_pri, "value")
pw_slider.observe(update_pri, "value")
display(widgets.VBox([prf_slider, pw_slider]), out2)
update_pri()

Output()

## 3 — The Chirp Signal (Linear Frequency Modulation)

A simple short pulse gives good range resolution but carries little energy.
A **chirp** sweeps the carrier frequency linearly over bandwidth $B$ during pulse duration $T$:

$$s(t) = \cos\!\Big(2\pi f_0\,t + \pi \frac{B}{T}\,t^2\Big), \quad 0 \le t \le T$$

The instantaneous frequency rises from $f_0$ to $f_0 + B$.
After reception the chirp is **pulse-compressed** by a matched filter,
giving the resolution of a short pulse ($c/2B$) with the energy of a long one.

In [5]:
out3 = widgets.Output()

bw_slider = widgets.FloatSlider(value=5, min=0.5, max=20, step=0.5,
    description="BW [MHz]:", style={"description_width": "100px"},
    layout=widgets.Layout(width="560px"))
dur_slider = widgets.FloatSlider(value=10, min=1, max=50, step=1,
    description="Duration [µs]:", style={"description_width": "100px"},
    layout=widgets.Layout(width="560px"))

def update_chirp(change=None):
    B = bw_slider.value * 1e6
    T = dur_slider.value * 1e-6
    f0 = 0
    n = 4000
    t = np.linspace(0, T, n)
    phase = 2 * np.pi * (f0 * t + 0.5 * (B / T) * t ** 2)
    sig = np.cos(phase)
    f_inst = f0 + (B / T) * t
    with out3:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
        axes[0].plot(t * 1e6, sig, lw=0.7, color="C0")
        axes[0].set_title("Chirp waveform")
        axes[0].set_xlabel("Time [µs]")
        axes[0].set_ylabel("Amplitude")
        axes[1].plot(t * 1e6, f_inst / 1e6, color="C1", lw=2)
        axes[1].set_title("Instantaneous frequency")
        axes[1].set_xlabel("Time [µs]")
        axes[1].set_ylabel("Freq [MHz]")
        fig.suptitle(f"BW = {B/1e6:.1f} MHz   T = {T*1e6:.0f} µs   Time-BW product = {B*T:.0f}",
                     fontsize=10, y=1.02)
        plt.tight_layout()
        plt.show()

bw_slider.observe(update_chirp, "value")
dur_slider.observe(update_chirp, "value")
display(widgets.VBox([bw_slider, dur_slider]), out3)
update_chirp()

Output()

## 4 — Matched Filter and Pulse Compression

The receiver correlates the echo with a copy of the transmitted chirp.
This **matched filter** compresses the long chirp into a narrow peak whose width ≈ $1/B$.

The **pulse compression ratio** is $B \cdot T$ (time–bandwidth product).
Larger → sharper peak → better range resolution with the same energy.

In [6]:
out4 = widgets.Output()

bw_mf = widgets.FloatSlider(value=5, min=0.5, max=20, step=0.5,
    description="BW [MHz]:", style={"description_width": "100px"},
    layout=widgets.Layout(width="560px"))
dur_mf = widgets.FloatSlider(value=10, min=2, max=50, step=1,
    description="Duration [µs]:", style={"description_width": "100px"},
    layout=widgets.Layout(width="560px"))

def update_mf(change=None):
    B = bw_mf.value * 1e6
    T = dur_mf.value * 1e-6
    n = 4000
    t = np.linspace(0, T, n)
    phase = 2 * np.pi * 0.5 * (B / T) * t ** 2
    chirp_sig = np.cos(phase)
    mf_out = correlate(chirp_sig, chirp_sig, mode="full")
    mf_out = mf_out / np.max(np.abs(mf_out))
    dt = T / n
    t_mf = np.arange(len(mf_out)) * dt - T
    res_range = c / (2 * B)
    with out4:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
        axes[0].plot(t * 1e6, chirp_sig, lw=0.6, color="C0")
        axes[0].set_title(f"Transmitted chirp  (T = {T*1e6:.0f} µs)")
        axes[0].set_xlabel("Time [µs]")
        axes[1].plot(t_mf * 1e6, mf_out, lw=1, color="C3")
        axes[1].set_title(f"After matched filter  |  ΔR = c/(2B) = {res_range:.1f} m")
        axes[1].set_xlabel("Time [µs]")
        axes[1].set_xlim(-3 * 1e6 / B, 3 * 1e6 / B)
        fig.suptitle(f"Compression ratio BT = {B*T:.0f}", fontsize=10, y=1.02)
        plt.tight_layout()
        plt.show()

bw_mf.observe(update_mf, "value")
dur_mf.observe(update_mf, "value")
display(widgets.VBox([bw_mf, dur_mf]), out4)
update_mf()

Output()

## 5 — Range Resolution: Can the Radar Tell Two Targets Apart?

Imagine two aircraft flying at nearly the same bearing but at slightly different distances.
The radar sends one pulse that bounces off both.
If the aircraft are **too close**, their echoes overlap and the radar sees **one blob**.
It can distinguish them only when:

$$\Delta R > \frac{c}{2B}$$

where $B$ is the signal bandwidth.

**Top panel** — a physical picture: the radar on the left, two targets on the right.
The pulse travels, bounces off both, and returns two echoes.
**Bottom panel** — what the radar actually "sees" after processing: the matched-filter output.
When the peaks merge, the targets are **unresolved**. Drag the separation slider to find the threshold.

In [7]:
out5 = widgets.Output()

bw_res = widgets.FloatSlider(value=2, min=0.5, max=10, step=0.25,
    description="BW [MHz]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))
r1_sl = widgets.FloatSlider(value=50, min=10, max=100, step=0.5,
    description="Target 1 R [km]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))
sep_sl = widgets.FloatSlider(value=200, min=0, max=500, step=5,
    description="Separation [m]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))

def update_res(change=None):
    B = bw_res.value * 1e6
    R1 = r1_sl.value * 1e3
    sep = sep_sl.value
    R2 = R1 + sep
    delta_r = c / (2 * B)
    resolved = sep >= delta_r

    T = 10e-6
    n = 6000
    t_ref = np.linspace(0, T, n)
    ref = np.cos(2 * np.pi * 0.5 * (B / T) * t_ref ** 2)
    dt1 = 2 * R1 / c
    dt2 = 2 * R2 / c
    t_full = np.linspace(dt1 - 2 * T, dt2 + 3 * T, 20000)
    rx = np.zeros_like(t_full)
    for delay, amp in [(dt1, 1.0), (dt2, 0.85)]:
        t_local = t_full - delay
        mask = (t_local >= 0) & (t_local <= T)
        rx[mask] += amp * np.cos(2 * np.pi * 0.5 * (B / T) * t_local[mask] ** 2)
    mf = correlate(rx, ref, mode="same")
    mf = np.abs(mf) / np.max(np.abs(mf))
    r_axis = c * (t_full - t_full[0]) / 2
    r_axis = r_axis - r_axis[len(r_axis) // 2] + R1 + sep / 2

    color = "C2" if resolved else "C3"
    label = "RESOLVED" if resolved else "NOT RESOLVED"

    with out5:
        clear_output(wait=True)
        fig, axes = plt.subplots(2, 1, figsize=(11, 6),
                                 gridspec_kw={"height_ratios": [1, 1.3]})

        ax_phys = axes[0]
        x_lo = R1 - 4 * delta_r
        x_hi = R2 + 4 * delta_r
        ax_phys.set_xlim(x_lo, x_hi)
        ax_phys.set_ylim(-1, 1)
        ax_phys.set_yticks([])
        ax_phys.axhline(0, color="gray", lw=0.4, alpha=0.4)
        ax_phys.plot(R1, 0.05, "D", color="C0", ms=14, zorder=5)
        ax_phys.annotate("T1", (R1, 0.35), ha="center", fontsize=10, color="C0", weight="bold")
        ax_phys.plot(R2, 0.05, "D", color="C1", ms=14, zorder=5)
        ax_phys.annotate("T2", (R2, 0.35), ha="center", fontsize=10, color="C1", weight="bold")
        ax_phys.annotate("", xy=(R2, -0.35), xytext=(R1, -0.35),
                         arrowprops=dict(arrowstyle="<->", color="k", lw=1.5))
        ax_phys.text((R1 + R2) / 2, -0.6, f"Δ = {sep:.0f} m",
                     ha="center", fontsize=10,
                     bbox=dict(fc="lightyellow", ec="gray", pad=3))
        ax_phys.axvspan(R1 - delta_r / 2, R1 + delta_r / 2,
                        alpha=0.12, color="C0", label=f"Resolution cell = {delta_r:.0f} m")
        ax_phys.axvspan(R2 - delta_r / 2, R2 + delta_r / 2, alpha=0.12, color="C1")
        ax_phys.legend(loc="upper right", fontsize=8)
        ax_phys.set_title(f"Physical view — two targets separated by {sep:.0f} m  [{label}]",
                          fontsize=11, color=color)
        ax_phys.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e3:.2f}"))
        ax_phys.set_xlabel("Range [km]")

        ax_mf = axes[1]
        ax_mf.plot(r_axis / 1e3, mf, color=color, lw=1.5)
        ax_mf.axvline(R1 / 1e3, ls="--", color="C0", lw=0.9, alpha=0.7, label="T1")
        ax_mf.axvline(R2 / 1e3, ls="--", color="C1", lw=0.9, alpha=0.7, label="T2")
        ax_mf.set_xlim((R1 - 3 * delta_r) / 1e3, (R2 + 3 * delta_r) / 1e3)
        ax_mf.set_xlabel("Range [km]")
        ax_mf.set_ylabel("|Matched filter|")
        ax_mf.set_title("Radar output — what the operator sees on screen", fontsize=10)
        ax_mf.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

bw_res.observe(update_res, "value")
r1_sl.observe(update_res, "value")
sep_sl.observe(update_res, "value")
display(widgets.VBox([bw_res, r1_sl, sep_sl]), out5)
update_res()

Output()

## 6 — The Doppler Effect: Measuring Velocity

When a car drives toward you honking its horn, the pitch sounds higher; as it drives away the pitch drops.
Radar waves behave the same way. A target approaching at velocity $v_r$ compresses the reflected wave,
shifting its frequency up by:

$$f_d = \frac{2\,v_r\,f_0}{c}$$

The radar measures this **Doppler shift** to compute the target's radial velocity.

The simulation below shows a **physical scene**: a radar on the left and a target moving along the range axis.
Use the **time slider** to watch the target move; the bottom panel shows the wavefronts
received by the radar getting compressed (approach) or stretched (recession).

In [8]:
out6 = widgets.Output()

vel_sl = widgets.FloatSlider(value=150, min=-400, max=400, step=5,
    description="Velocity [m/s]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))
t6_sl = widgets.FloatSlider(value=0, min=0, max=4.0, step=0.05,
    description="Time [s]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))

def update_doppler_phys(change=None):
    vr = vel_sl.value
    t_now = t6_sl.value
    R0 = 5000
    f0 = 10e9
    lam = c / f0
    fd = 2 * vr * f0 / c
    R_now = R0 - vr * t_now
    R_min = R0 - abs(vr) * 4 - 500
    R_max_plot = R0 + abs(vr) * 4 + 500
    if vr > 0:
        direction = "← approaching"
        dir_color = "C2"
    elif vr < 0:
        direction = "receding →"
        dir_color = "C3"
    else:
        direction = "stationary"
        dir_color = "gray"

    with out6:
        clear_output(wait=True)
        fig, axes = plt.subplots(2, 1, figsize=(11, 5.5),
                                 gridspec_kw={"height_ratios": [1.1, 1]})

        ax = axes[0]
        ax.set_xlim(-200, max(R_max_plot, R0 + 500))
        ax.set_ylim(-1.2, 1.2)
        ax.set_yticks([])
        ax.axhline(0, color="gray", lw=0.3)
        ax.plot(0, 0, "s", color="C0", ms=18, zorder=5)
        ax.annotate("RADAR", (0, -0.7), ha="center", fontsize=9, color="C0", weight="bold")
        ax.plot(max(R_now, 100), 0, ">", color=dir_color, ms=16, zorder=5)
        ax.annotate(f"TARGET\nR = {max(R_now,0):.0f} m\n{direction}",
                    (max(R_now, 100), 0.55), ha="center", fontsize=9, color=dir_color)
        n_fronts = 8
        for i in range(n_fronts):
            r_front = max(R_now, 100) - i * lam * 1e5 * (1 + fd / f0)
            if 0 < r_front < max(R_now, 100):
                alpha = 0.6 - 0.06 * i
                ax.axvline(r_front, color=dir_color, alpha=max(alpha, 0.1), lw=1)
        ax.set_xlabel("Distance from radar [m]")
        ax.set_title(f"t = {t_now:.2f} s  |  v = {vr:.0f} m/s  |  f_d = {fd:.0f} Hz  ({fd/1e3:.2f} kHz)",
                     fontsize=11)
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}"))

        ax2 = axes[1]
        t_sig = np.linspace(0, 4, 3000)
        n_cycles_show = 10
        T_show = n_cycles_show / max(abs(fd), 1)
        t_wave = np.linspace(0, min(T_show, 0.05), 3000)
        tx_wave = np.cos(2 * np.pi * 500 * t_wave)
        rx_wave = np.cos(2 * np.pi * (500 + fd / (f0 / 500)) * t_wave)
        ax2.plot(t_wave * 1e3, tx_wave, color="C0", lw=1.2, alpha=0.6, label="Tx reference")
        ax2.plot(t_wave * 1e3, rx_wave, color=dir_color, lw=1.5, label="Rx (Doppler shifted)")
        ax2.set_xlabel("Time [ms] (baseband representation)")
        ax2.set_ylabel("Amplitude")
        ax2.set_title("Received signal vs transmitted — frequency difference = Doppler shift")
        ax2.legend(loc="upper right", fontsize=8)
        plt.tight_layout()
        plt.show()

vel_sl.observe(update_doppler_phys, "value")
t6_sl.observe(update_doppler_phys, "value")
display(widgets.VBox([vel_sl, t6_sl]), out6)
update_doppler_phys()

Output()

## 7 — Moving-Target Tracking: Pulses over Time

The radar fires a train of pulses. For each pulse it records which range bin lights up.
Stacking all the returns in order gives a **Range–Time Intensity** (RTI) plot:
range on the horizontal axis, pulse number on the vertical.
A moving target draws a diagonal line — its slope reveals the velocity.

Adjust the target's speed and initial range to see the track change.

In [9]:
out7 = widgets.Output()

v_sim = widgets.FloatSlider(value=300, min=-600, max=600, step=10,
    description="Velocity [m/s]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))
r0_sim = widgets.FloatSlider(value=50, min=10, max=150, step=1,
    description="Initial R [km]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))
prf_sim = widgets.FloatSlider(value=1000, min=200, max=3000, step=100,
    description="PRF [Hz]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))

def update_sim(change=None):
    vr = v_sim.value
    R0 = r0_sim.value * 1e3
    prf = prf_sim.value
    pri = 1 / prf
    N = 40
    f0 = 10e9
    lam = c / f0
    B = 2e6
    R_max = c * pri / 2
    range_bins = np.linspace(0, R_max, 800)
    delta_r = c / (2 * B)
    ranges = np.zeros(N)
    range_profiles = np.zeros((N, len(range_bins)))
    for i in range(N):
        t_pulse = i * pri
        R_cur = R0 + vr * t_pulse
        ranges[i] = R_cur
        profile = np.exp(-0.5 * ((range_bins - R_cur) / (delta_r / 2)) ** 2)
        range_profiles[i] = profile
    fd = 2 * vr * f0 / c
    pulse_idx = np.arange(N)
    with out7:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
        r_lo = max(0, min(ranges) - 5e3)
        r_hi = max(ranges) + 5e3
        if r_lo >= r_hi:
            r_lo, r_hi = R0 - 5e3, R0 + 5e3
        lo_idx = np.searchsorted(range_bins, r_lo)
        hi_idx = np.searchsorted(range_bins, r_hi)
        extent = [range_bins[lo_idx] / 1e3, range_bins[min(hi_idx, len(range_bins)-1)] / 1e3,
                  N - 0.5, -0.5]
        axes[0].imshow(range_profiles[:, lo_idx:hi_idx], aspect="auto",
                       extent=extent, cmap="inferno")
        axes[0].set_xlabel("Range [km]")
        axes[0].set_ylabel("Pulse #")
        axes[0].set_title("Range–Time Intensity (RTI)")
        axes[1].plot(pulse_idx, ranges / 1e3, "o-", ms=3, color="C0")
        slope_kmps = vr / 1e3
        axes[1].set_xlabel("Pulse #")
        axes[1].set_ylabel("Range [km]")
        axes[1].set_title(f"Track: v = {vr:.0f} m/s  |  f_d = {fd:.0f} Hz")
        plt.tight_layout()
        plt.show()

for w in [v_sim, r0_sim, prf_sim]:
    w.observe(update_sim, "value")
display(widgets.VBox([v_sim, r0_sim, prf_sim]), out7)
update_sim()

Output()

## 8 — The PRF Dilemma: Range vs Velocity Ambiguity

$$R_{\max} = \frac{c}{2\,\text{PRF}}, \qquad v_{\max} = \frac{\lambda\,\text{PRF}}{4}$$

These two limits are **inversely coupled**:
*low PRF* → long listening window → clear in range but ambiguous in velocity;
*high PRF* → frequent updates → clear in velocity but ambiguous in range.

In [10]:
out8 = widgets.Output()

prf_amb = widgets.FloatSlider(value=2000, min=100, max=20000, step=100,
    description="PRF [Hz]:", style={"description_width": "100px"},
    layout=widgets.Layout(width="560px"))
fc_amb = widgets.FloatSlider(value=10, min=1, max=35, step=0.5,
    description="Carrier [GHz]:", style={"description_width": "100px"},
    layout=widgets.Layout(width="560px"))

def update_ambiguity(change=None):
    prf = prf_amb.value
    f0 = fc_amb.value * 1e9
    lam = c / f0
    r_max = c / (2 * prf)
    v_max = lam * prf / 4
    prf_range = np.logspace(2, 5, 500)
    r_arr = c / (2 * prf_range) / 1e3
    v_arr = lam * prf_range / 4
    with out8:
        clear_output(wait=True)
        fig, ax1 = plt.subplots(figsize=(10, 4))
        ax1.semilogx(prf_range, r_arr, color="C0", lw=2, label="R_max")
        ax1.axvline(prf, ls=":", color="gray", lw=1)
        ax1.plot(prf, r_max / 1e3, "o", color="C0", ms=10, zorder=5)
        ax1.set_xlabel("PRF [Hz]")
        ax1.set_ylabel("R_max [km]", color="C0")
        ax1.tick_params(axis="y", labelcolor="C0")
        ax2 = ax1.twinx()
        ax2.semilogx(prf_range, v_arr, color="C3", lw=2, label="v_max")
        ax2.plot(prf, v_max, "s", color="C3", ms=10, zorder=5)
        ax2.set_ylabel("v_max [m/s]", color="C3")
        ax2.tick_params(axis="y", labelcolor="C3")
        ax1.set_title(f"PRF = {prf:.0f} Hz  →  R_max = {r_max/1e3:.1f} km,  "
                      f"v_max = {v_max:.1f} m/s  ({v_max*3.6:.0f} km/h)")
        fig.legend(loc="upper center", ncol=2, fontsize=9, bbox_to_anchor=(0.5, 0.92))
        plt.tight_layout()
        plt.show()

prf_amb.observe(update_ambiguity, "value")
fc_amb.observe(update_ambiguity, "value")
display(widgets.VBox([prf_amb, fc_amb]), out8)
update_ambiguity()

Output()

## 9 — Range–Doppler Map

By stacking many pulses and applying an FFT across the pulse dimension,
the radar builds a 2-D image with **range** on one axis and **velocity** on the other.
Each target shows as a bright spot at its $(R, v)$ coordinate.
Place up to three targets and see them light up on the map.

In [ ]:
out9 = widgets.Output()

n_tgt = widgets.IntSlider(value=2, min=1, max=3, step=1,
    description="Targets:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))
r_t1 = widgets.FloatSlider(value=30, min=5, max=80, step=1,
    description="T1 range [km]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))
v_t1 = widgets.FloatSlider(value=200, min=-500, max=500, step=10,
    description="T1 vel [m/s]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))
r_t2 = widgets.FloatSlider(value=45, min=5, max=80, step=1,
    description="T2 range [km]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))
v_t2 = widgets.FloatSlider(value=-150, min=-500, max=500, step=10,
    description="T2 vel [m/s]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))
r_t3 = widgets.FloatSlider(value=30, min=5, max=80, step=1,
    description="T3 range [km]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))
v_t3 = widgets.FloatSlider(value=50, min=-500, max=500, step=10,
    description="T3 vel [m/s]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))

def update_rd(change=None):
    f0 = 10e9
    lam = c / f0
    prf = 2000
    pri = 1 / prf
    N = 64
    B = 2e6
    delta_r = c / (2 * B)
    n_range = 512
    targets = [(r_t1.value * 1e3, v_t1.value)]
    if n_tgt.value >= 2:
        targets.append((r_t2.value * 1e3, v_t2.value))
    if n_tgt.value >= 3:
        targets.append((r_t3.value * 1e3, v_t3.value))
    data = np.zeros((N, n_range), dtype=complex)
    range_axis = np.linspace(0, c * pri / 2, n_range)
    for R, v in targets:
        fd = 2 * v / lam
        r_idx = np.argmin(np.abs(range_axis - R))
        spread = int(max(1, delta_r / (range_axis[1] - range_axis[0])))
        for pulse_i in range(N):
            phase = 2 * np.pi * fd * pulse_i * pri
            lo = max(0, r_idx - spread)
            hi = min(n_range, r_idx + spread + 1)
            gauss = np.exp(-0.5 * ((np.arange(lo, hi) - r_idx) / max(spread / 2, 1)) ** 2)
            data[pulse_i, lo:hi] += gauss * np.exp(1j * phase)
    noise = 0.05 * (np.random.randn(N, n_range) + 1j * np.random.randn(N, n_range))
    data += noise
    rd_map = np.fft.fftshift(np.fft.fft(data, axis=0), axes=0)
    rd_power = 20 * np.log10(np.abs(rd_map) + 1e-12)
    rd_power -= rd_power.max()
    vel_axis = np.fft.fftshift(np.fft.fftfreq(N, d=pri)) * lam / 2
    with out9:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 5))
        extent = [range_axis[0] / 1e3, range_axis[-1] / 1e3,
                  vel_axis[0], vel_axis[-1]]
        ax.imshow(rd_power, aspect="auto", extent=extent, origin="lower",
                  cmap="viridis", vmin=-30, vmax=0)
        for i, (R, v) in enumerate(targets):
            ax.plot(R / 1e3, v, "rx", ms=12, mew=2)
            ax.annotate(f"T{i+1}", (R / 1e3, v), fontsize=9,
                        color="white", xytext=(5, 5), textcoords="offset points")
        ax.set_xlabel("Range [km]")
        ax.set_ylabel("Velocity [m/s]")
        ax.set_title("Range–Doppler Map  (dB, normalised)")
        plt.tight_layout()
        plt.show()

for w in [n_tgt, r_t1, v_t1, r_t2, v_t2, r_t3, v_t3]:
    w.observe(update_rd, "value")
display(widgets.VBox([n_tgt, r_t1, v_t1, r_t2, v_t2, r_t3, v_t3]), out9)
update_rd()

Output()

## 10 — Radar Operating Modes: Search, Track, TWS

| Mode | Beam behaviour | Purpose |
|---|---|---|
| **Search** | Wide beam sweeps 360° (or a sector) | Detect new targets |
| **Track** | Narrow beam locked on one target | Precise position & velocity |
| **Track-While-Scan** | Scans but dwells longer on known targets | Both at once |

The top panel shows a **plan view** — radar at the centre, targets as triangles,
and the beam as a shaded cone. Targets drift in real time.
The bottom panel shows the **pulse timeline**: each vertical bar is one transmitted pulse.
In Track mode the PRF is higher (more updates); in Search the beam sweeps wider.

In [ ]:
out10 = widgets.Output()

mode_dd = widgets.Dropdown(
    options=["Search (360°)", "Sector Search (±45°)",
             "Track (single target)", "Track-While-Scan"],
    value="Search (360°)",
    description="Radar mode:", style={"description_width": "100px"},
    layout=widgets.Layout(width="360px"))
step_sl = widgets.IntSlider(value=0, min=0, max=119, step=1,
    description="Time step:", style={"description_width": "100px"},
    layout=widgets.Layout(width="560px"))

np.random.seed(42)
_tgt_r0 = np.array([35, 55, 25, 60])
_tgt_az0 = np.array([30, -20, 70, -50], dtype=float)
_tgt_vr = np.array([-0.8, 0.5, -0.3, 1.0])
_tgt_vaz = np.array([0.4, -0.3, 0.6, -0.2])

def update_mode(change=None):
    mode = mode_dd.value
    step = step_sl.value
    t_step = step * 0.1
    tgt_r = _tgt_r0 + _tgt_vr * t_step
    tgt_az = _tgt_az0 + _tgt_vaz * t_step
    R_max_plot = 80
    with out10:
        clear_output(wait=True)
        fig = plt.figure(figsize=(11, 5.5))
        ax_polar = fig.add_subplot(121, projection="polar")
        ax_pulse = fig.add_subplot(122)

        ax_polar.set_ylim(0, R_max_plot)
        ax_polar.set_theta_zero_location("N")
        ax_polar.set_theta_direction(-1)

        for i in range(len(tgt_r)):
            ang = np.deg2rad(tgt_az[i])
            ax_polar.plot(ang, tgt_r[i], "r^", ms=9, zorder=5)
            ax_polar.annotate(f"T{i+1}", (ang, tgt_r[i]),
                              fontsize=7, color="red", xytext=(3, 3),
                              textcoords="offset points")

        if mode == "Search (360°)":
            beam_w = np.deg2rad(15)
            az = np.deg2rad(step * 3 % 360)
            prf_mode = 500
            theta_fill = np.linspace(az - beam_w / 2, az + beam_w / 2, 50)
            ax_polar.fill_between(theta_fill, 0, R_max_plot, alpha=0.25, color="C0")
            ax_polar.plot([az, az], [0, R_max_plot], color="C0", lw=2)
            ax_polar.set_title(f"Search  az={np.rad2deg(az):.0f}°", va="bottom", fontsize=10)
        elif mode == "Sector Search (±45°)":
            beam_w = np.deg2rad(8)
            frac = (step % 60) / 60
            if (step // 60) % 2 == 1:
                frac = 1 - frac
            az = np.deg2rad(-45 + 90 * frac)
            prf_mode = 800
            theta_fill = np.linspace(az - beam_w / 2, az + beam_w / 2, 50)
            ax_polar.fill_between(theta_fill, 0, R_max_plot, alpha=0.25, color="C1")
            ax_polar.plot([az, az], [0, R_max_plot], color="C1", lw=2)
            ax_polar.set_title(f"Sector ±45°  az={np.rad2deg(az):.0f}°", va="bottom", fontsize=10)
        elif mode == "Track (single target)":
            tgt_idx = 0
            az = np.deg2rad(tgt_az[tgt_idx])
            beam_w = np.deg2rad(3)
            prf_mode = 3000
            theta_fill = np.linspace(az - beam_w / 2, az + beam_w / 2, 50)
            ax_polar.fill_between(theta_fill, 0, R_max_plot, alpha=0.3, color="C2")
            ax_polar.plot([az, az], [0, R_max_plot], color="C2", lw=2)
            ax_polar.set_title(f"Track  locked T1", va="bottom", fontsize=10)
        elif mode == "Track-While-Scan":
            is_dwell = step % 10 < 3
            prf_mode = 1500
            if is_dwell:
                tgt_idx = (step // 10) % len(tgt_r)
                az = np.deg2rad(tgt_az[tgt_idx])
                beam_w = np.deg2rad(4)
                theta_fill = np.linspace(az - beam_w / 2, az + beam_w / 2, 50)
                ax_polar.fill_between(theta_fill, 0, R_max_plot, alpha=0.35, color="C2")
                ax_polar.plot([az, az], [0, R_max_plot], color="C2", lw=2)
                ax_polar.set_title(f"TWS  dwell T{tgt_idx+1}", va="bottom", fontsize=10)
            else:
                az = np.deg2rad(step * 3 % 360)
                beam_w = np.deg2rad(12)
                theta_fill = np.linspace(az - beam_w / 2, az + beam_w / 2, 50)
                ax_polar.fill_between(theta_fill, 0, R_max_plot, alpha=0.2, color="C0")
                ax_polar.plot([az, az], [0, R_max_plot], color="C0", lw=2)
                ax_polar.set_title("TWS  scanning", va="bottom", fontsize=10)

        pri_mode = 1 / prf_mode
        n_show = min(20, int(0.02 / pri_mode))
        t_pulses = np.arange(n_show) * pri_mode
        for tp in t_pulses:
            ax_pulse.axvline(tp * 1e3, color="C0", lw=2, alpha=0.7)
        ax_pulse.set_xlim(-0.5, t_pulses[-1] * 1e3 + 1 if n_show > 0 else 5)
        ax_pulse.set_ylim(0, 1.2)
        ax_pulse.set_xlabel("Time [ms]")
        ax_pulse.set_ylabel("Pulse")
        ax_pulse.set_title(f"Pulse train  PRF = {prf_mode} Hz  |  PRI = {pri_mode*1e6:.0f} µs")
        ax_pulse.set_yticks([])
        plt.tight_layout()
        plt.show()

mode_dd.observe(update_mode, "value")
step_sl.observe(update_mode, "value")
display(widgets.VBox([mode_dd, step_sl]), out10)
update_mode()

Output()

## 11 — Radar Range Equation

$$P_r = \frac{P_t\, G^2\, \lambda^2\, \sigma}{(4\pi)^3\, R^4}$$

The $R^4$ law makes distant targets extremely faint. The **maximum detection range** is:

$$R_{\max} = \left(\frac{P_t\, G^2\, \lambda^2\, \sigma}{(4\pi)^3\, S_{\min}}\right)^{1/4}$$

The left plot shows received power vs range. The right plot shows a **physical scenario**:
a radar tries to detect a target — when the target is beyond $R_{\max}$, it fades out.

In [12]:
out11 = widgets.Output()

pt_sl = widgets.FloatLogSlider(value=1e3, base=10, min=1, max=7, step=0.25,
    description="Pt [W]:", style={"description_width": "110px"},
    layout=widgets.Layout(width="560px"))
gain_sl = widgets.FloatSlider(value=30, min=10, max=50, step=1,
    description="Gain [dBi]:", style={"description_width": "110px"},
    layout=widgets.Layout(width="560px"))
rcs_sl = widgets.FloatLogSlider(value=1, base=10, min=-2, max=2, step=0.25,
    description="RCS σ [m²]:", style={"description_width": "110px"},
    layout=widgets.Layout(width="560px"))
tgt_r_eq = widgets.FloatSlider(value=80, min=5, max=250, step=1,
    description="Target R [km]:", style={"description_width": "110px"},
    layout=widgets.Layout(width="560px"))

def update_req(change=None):
    Pt = pt_sl.value
    G_dB = gain_sl.value
    G = 10 ** (G_dB / 10)
    sigma = rcs_sl.value
    f0 = 10e9
    lam = c / f0
    S_min = 1e-13
    R_max = (Pt * G ** 2 * lam ** 2 * sigma / ((4 * np.pi) ** 3 * S_min)) ** 0.25
    R_tgt = tgt_r_eq.value * 1e3
    detected = R_tgt <= R_max
    R = np.linspace(1e3, 300e3, 1000)
    Pr = Pt * G ** 2 * lam ** 2 * sigma / ((4 * np.pi) ** 3 * R ** 4)
    Pr_dBm = 10 * np.log10(Pr * 1e3 + 1e-30)
    S_dBm = 10 * np.log10(S_min * 1e3)
    with out11:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

        ax = axes[0]
        ax.plot(R / 1e3, Pr_dBm, color="C0", lw=2)
        ax.axhline(S_dBm, color="C3", ls="--", lw=1.5, label=f"S_min = {S_dBm:.0f} dBm")
        if R_max < R[-1]:
            ax.axvline(R_max / 1e3, color="C2", ls=":", lw=1.5,
                       label=f"R_max = {R_max/1e3:.1f} km")
        ax.axvline(R_tgt / 1e3, color="orange", lw=1.5, ls="-.", label=f"Target = {R_tgt/1e3:.0f} km")
        ax.set_xlabel("Range [km]")
        ax.set_ylabel("Received power [dBm]")
        ax.set_title("Power vs range")
        ax.legend(fontsize=8)
        ax.set_ylim(S_dBm - 20, Pr_dBm[0] + 5)

        ax2 = axes[1]
        ax2.set_xlim(-5, max(R_max / 1e3 * 1.3, R_tgt / 1e3 * 1.1))
        ax2.set_ylim(-1, 1)
        ax2.set_yticks([])
        ax2.axhline(0, color="gray", lw=0.3)
        ax2.plot(0, 0, "s", color="C0", ms=18, zorder=5)
        ax2.annotate("RADAR", (0, -0.5), ha="center", fontsize=9, color="C0", weight="bold")
        if R_max / 1e3 < ax2.get_xlim()[1]:
            ax2.axvline(R_max / 1e3, color="C2", ls=":", lw=2)
            ax2.annotate(f"R_max = {R_max/1e3:.1f} km", (R_max / 1e3, 0.8),
                         fontsize=9, color="C2", ha="center")
        tgt_alpha = 1.0 if detected else 0.25
        tgt_color = "C2" if detected else "C3"
        ax2.plot(R_tgt / 1e3, 0, "D", color=tgt_color, ms=16, alpha=tgt_alpha, zorder=5)
        status = "DETECTED" if detected else "BELOW THRESHOLD"
        ax2.annotate(f"TARGET  [{status}]", (R_tgt / 1e3, -0.5), ha="center",
                     fontsize=10, color=tgt_color, weight="bold")
        ax2.set_xlabel("Range [km]")
        ax2.set_title("Physical scenario", fontsize=11)
        plt.tight_layout()
        plt.show()

for w in [pt_sl, gain_sl, rcs_sl, tgt_r_eq]:
    w.observe(update_req, "value")
display(widgets.VBox([pt_sl, gain_sl, rcs_sl, tgt_r_eq]), out11)
update_req()

Output()

## 12 — Clutter Rejection with MTI Filtering

Ground, buildings and rain produce strong echoes near zero Doppler.
**Moving Target Indication (MTI)** subtracts consecutive pulses to cancel these static returns:

$$y[n] = x[n] - x[n-1]$$

The left panel shows the filter's frequency response — a null at zero Doppler (stationary clutter suppressed).
The right panel shows a **before / after scenario**: a range profile with a strong clutter peak and a weaker
moving target. After MTI, the clutter vanishes and the target stands out.

In [ ]:
out12 = widgets.Output()

prf_mti = widgets.FloatSlider(value=2000, min=500, max=5000, step=100,
    description="PRF [Hz]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))
clutter_sl = widgets.FloatSlider(value=30, min=5, max=80, step=1,
    description="Clutter R [km]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))
tgt_r_mti = widgets.FloatSlider(value=45, min=5, max=80, step=1,
    description="Target R [km]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))
tgt_v_mti = widgets.FloatSlider(value=200, min=10, max=500, step=10,
    description="Target v [m/s]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))

def update_mti(change=None):
    prf = prf_mti.value
    pri = 1 / prf
    f0 = 10e9
    lam = c / f0
    R_clutter = clutter_sl.value * 1e3
    R_tgt = tgt_r_mti.value * 1e3
    v_tgt = tgt_v_mti.value
    fd_tgt = 2 * v_tgt / lam

    f = np.linspace(-prf / 2, prf / 2, 2000)
    H = 2 * np.abs(np.sin(np.pi * f / prf))
    H_dB = 20 * np.log10(H + 1e-12)

    range_axis = np.linspace(0, c * pri / 2, 1000)
    dr = 500
    clutter_profile = 10 * np.exp(-0.5 * ((range_axis - R_clutter) / dr) ** 2)
    tgt_profile = 2 * np.exp(-0.5 * ((range_axis - R_tgt) / (dr / 3)) ** 2)
    before = clutter_profile + tgt_profile + 0.3 * np.random.randn(len(range_axis))
    gain_clutter = 2 * np.abs(np.sin(np.pi * 0 / prf))
    gain_tgt = 2 * np.abs(np.sin(np.pi * fd_tgt / prf))
    after = clutter_profile * gain_clutter + tgt_profile * gain_tgt + 0.3 * np.random.randn(len(range_axis))

    with out12:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(f / 1e3, H_dB, color="C0", lw=2)
        axes[0].axvline(fd_tgt / 1e3, color="C2", ls="--", lw=1,
                        label=f"Target f_d = {fd_tgt:.0f} Hz")
        axes[0].axvline(0, color="C3", ls=":", lw=1, label="Clutter (0 Hz)")
        axes[0].set_xlabel("Doppler [kHz]")
        axes[0].set_ylabel("|H| [dB]")
        axes[0].set_title("MTI filter response")
        axes[0].set_ylim(-50, 10)
        axes[0].legend(fontsize=8)

        axes[1].plot(range_axis / 1e3, before, color="gray", alpha=0.6, lw=1, label="Before MTI")
        axes[1].plot(range_axis / 1e3, np.clip(after, 0, None), color="C2", lw=1.5, label="After MTI")
        axes[1].axvline(R_clutter / 1e3, color="C3", ls=":", alpha=0.5)
        axes[1].annotate("clutter", (R_clutter / 1e3, 9), fontsize=8, color="C3", ha="center")
        axes[1].axvline(R_tgt / 1e3, color="C2", ls=":", alpha=0.5)
        axes[1].annotate("target", (R_tgt / 1e3, 3), fontsize=8, color="C2", ha="center")
        axes[1].set_xlabel("Range [km]")
        axes[1].set_ylabel("Amplitude")
        axes[1].set_title("Range profile: before & after MTI")
        axes[1].legend(fontsize=8)
        plt.tight_layout()
        plt.show()

for w in [prf_mti, clutter_sl, tgt_r_mti, tgt_v_mti]:
    w.observe(update_mti, "value")
display(widgets.VBox([prf_mti, clutter_sl, tgt_r_mti, tgt_v_mti]), out12)
update_mti()

Output()

## 13 — Antenna Beamwidth and Angular Resolution

$$\theta_{3\text{dB}} \approx \frac{0.886\,\lambda}{D}$$

A larger aperture or higher frequency narrows the beam, improving the ability to separate
targets at the same range but different angles.
The **cross-range resolution** at range $R$ is $\Delta x_\perp = R\,\theta_{3\text{dB}}$.

In [1]:
out13 = widgets.Output()

D_sl = widgets.FloatSlider(value=2, min=0.2, max=10, step=0.1,
    description="Aperture D [m]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))
fc_bw = widgets.FloatSlider(value=10, min=1, max=35, step=0.5,
    description="Freq [GHz]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))
r_bw = widgets.FloatSlider(value=50, min=5, max=200, step=5,
    description="Range [km]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))

def update_beam(change=None):
    D = D_sl.value
    f0 = fc_bw.value * 1e9
    lam = c / f0
    R = r_bw.value * 1e3
    theta_3dB = 0.886 * lam / D
    theta_deg = np.rad2deg(theta_3dB)
    cross_res = R * theta_3dB
    theta = np.linspace(-np.pi / 6, np.pi / 6, 1000)
    u = np.pi * D / lam * np.sin(theta)
    pattern = np.where(np.abs(u) < 1e-10, 1.0, np.abs(np.sin(u) / u))
    pattern_dB = 20 * np.log10(pattern + 1e-12)
    with out13:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
        axes[0].plot(np.rad2deg(theta), pattern_dB, color="C0", lw=2)
        axes[0].axhline(-3, color="C3", ls="--", lw=1, label="-3 dB")
        axes[0].axvline(theta_deg / 2, color="gray", ls=":", lw=0.8)
        axes[0].axvline(-theta_deg / 2, color="gray", ls=":", lw=0.8)
        axes[0].set_xlabel("Angle [°]")
        axes[0].set_ylabel("Pattern [dB]")
        axes[0].set_ylim(-40, 3)
        axes[0].set_title(f"θ_3dB = {theta_deg:.2f}°  (D={D:.1f}m, λ={lam*100:.1f}cm)")
        axes[0].legend(fontsize=8)
        r_axis = np.linspace(0, R * 1.5, 200)
        half_w = r_axis * np.tan(theta_3dB / 2)
        axes[1].fill_betweenx(r_axis / 1e3, -half_w, half_w, alpha=0.2, color="C0")
        axes[1].plot(-half_w, r_axis / 1e3, color="C0", lw=1)
        axes[1].plot(half_w, r_axis / 1e3, color="C0", lw=1)
        axes[1].axhline(R / 1e3, color="C3", ls="--", lw=1)
        axes[1].annotate(f"Δx = {cross_res:.0f} m", xy=(cross_res * 0.5, R / 1e3),
                         fontsize=10, color="C3",
                         xytext=(cross_res, R / 1e3 + R / 1e3 * 0.08))
        axes[1].set_xlabel("Cross-range [m]")
        axes[1].set_ylabel("Range [km]")
        axes[1].set_title(f"Beam footprint  |  Δx at {R/1e3:.0f} km = {cross_res:.0f} m")
        axes[1].set_xlim(-R * theta_3dB * 2, R * theta_3dB * 2)
        plt.tight_layout()
        plt.show()

for w in [D_sl, fc_bw, r_bw]:
    w.observe(update_beam, "value")
display(widgets.VBox([D_sl, fc_bw, r_bw]), out13)
update_beam()

NameError: name 'widgets' is not defined

## 14 — PPI Display with Moving Targets

The **Plan Position Indicator** is the classic circular radar screen.
The antenna sweeps 360° and paints detected targets as bright dots.
Targets drift slowly; their old detections fade like a phosphor afterglow.
Rotate the antenna with the slider and watch the scene evolve.

In [15]:
out14 = widgets.Output()

az_ppi = widgets.FloatSlider(value=0, min=0, max=355, step=5,
    description="Antenna az [°]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))
r_max_ppi = widgets.FloatSlider(value=100, min=20, max=200, step=10,
    description="Display R [km]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="560px"))

np.random.seed(7)
_n_t = 8
_tgt_r = np.random.uniform(15, 90, _n_t)
_tgt_az = np.random.uniform(0, 360, _n_t)
_tgt_v = np.random.uniform(-2, 2, _n_t)

def update_ppi(change=None):
    current_az = az_ppi.value
    R_max = r_max_ppi.value
    beam_w = 8
    tgt_az_now = (_tgt_az + _tgt_v * current_az / 5) % 360
    with out14:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(6.5, 6.5), subplot_kw={"projection": "polar"})
        ax.set_theta_zero_location("N")
        ax.set_theta_direction(-1)
        ax.set_ylim(0, R_max)
        beam_az = np.deg2rad(current_az)
        bw_rad = np.deg2rad(beam_w)
        theta_fill = np.linspace(beam_az - bw_rad / 2, beam_az + bw_rad / 2, 50)
        ax.fill_between(theta_fill, 0, R_max, alpha=0.15, color="lime")
        ax.plot([beam_az, beam_az], [0, R_max], color="lime", lw=1.5, alpha=0.6)
        trail_len = 14
        for step_back in range(trail_len):
            old_az = current_az - step_back * 5
            old_tgt_az = (_tgt_az + _tgt_v * old_az / 5) % 360
            old_diff = np.abs((old_tgt_az - old_az + 180) % 360 - 180)
            old_det = old_diff < beam_w / 2
            alpha_trail = 0.6 * (1 - step_back / trail_len)
            if np.any(old_det):
                ax.plot(np.deg2rad(old_tgt_az[old_det]), _tgt_r[old_det],
                        "o", color="lime", ms=4, alpha=max(alpha_trail, 0.05))
        az_diff = np.abs((tgt_az_now - current_az + 180) % 360 - 180)
        detected = az_diff < beam_w / 2
        if np.any(detected):
            ax.plot(np.deg2rad(tgt_az_now[detected]), _tgt_r[detected],
                    "o", color="lime", ms=8, alpha=0.95)
        ax.set_facecolor("#001a00")
        fig.patch.set_facecolor("#0a0a0a")
        ax.tick_params(colors="lime")
        ax.grid(color="lime", alpha=0.15)
        ax.spines["polar"].set_color("lime")
        ax.set_title("PPI Display", va="bottom", fontsize=12, color="lime")
        plt.tight_layout()
        plt.show()

az_ppi.observe(update_ppi, "value")
r_max_ppi.observe(update_ppi, "value")
display(widgets.VBox([az_ppi, r_max_ppi]), out14)
update_ppi()

Output()